In [1]:
"""
Tune four win-probability models: logistic regression, single classification tree,
random forest, and XGBoost.

Train: 2021-2023, Validation: 2024, Test: 2025.
"""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

CSV_PATH = "/Users/firea/Downloads/16_win-probability.csv"

# ---------------------------------------------------------------------------
# Load and split data
# ---------------------------------------------------------------------------
df = pd.read_csv("16_win-probability.csv.gz")

train = df[df["season"].between(2021, 2023)].copy()
val = df[df["season"] == 2024].copy()
test = df[df["season"] == 2025].copy()

n_train = len(train)
print(f"Train rows: {n_train}, Val rows: {len(val)}, Test rows: {len(test)}")

# ---------------------------------------------------------------------------
# Task 1: Logistic regression
# posteam_win ~ score_differential*game_seconds_remaining + yardline_100*down
#   + ydstogo + qtr + posteam_timeouts_remaining + defteam_timeouts_remaining
#   + posteam_spread + posteam_type
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 1: Logistic Regression (no tuning)")
print("=" * 70)

def build_logit_design(data):
    out = pd.DataFrame(index=data.index)
    out["score_differential"] = data["score_differential"]
    out["game_seconds_remaining"] = data["game_seconds_remaining"]
    out["score_diff_x_secs"] = data["score_differential"] * data["game_seconds_remaining"]
    out["yardline_100"] = data["yardline_100"]
    out["down"] = data["down"]
    out["yardline_x_down"] = data["yardline_100"] * data["down"]
    out["ydstogo"] = data["ydstogo"]
    out["qtr"] = data["qtr"]
    out["posteam_timeouts_remaining"] = data["posteam_timeouts_remaining"]
    out["defteam_timeouts_remaining"] = data["defteam_timeouts_remaining"]
    out["posteam_spread"] = data["posteam_spread"]
    out["posteam_type_home"] = (data["posteam_type"] == "home").astype(int)
    return out

X_train_logit = build_logit_design(train)
X_val_logit = build_logit_design(val)
X_test_logit = build_logit_design(test)

y_train = train["posteam_win"].astype(int)
y_val = val["posteam_win"].astype(int)
y_test = test["posteam_win"].astype(int)

logit_model = LogisticRegression(penalty=None, max_iter=2000)
logit_model.fit(X_train_logit, y_train)

val_pred_logit = logit_model.predict_proba(X_val_logit)[:, 1]
val_logloss_logit = log_loss(y_val, val_pred_logit)
print(f"Validation log loss: {val_logloss_logit:.5f}")

test_pred_logit = logit_model.predict_proba(X_test_logit)[:, 1]
test_logloss_logit = log_loss(y_test, test_pred_logit)
print(f"Test log loss: {test_logloss_logit:.5f}")

# Predictor set used by trees/forest/xgboost (raw versions of the variables
# that appear in the logistic formula; tree-based models learn interactions
# on their own so no manual interaction terms are added).
PREDICTORS = [
    "score_differential",
    "game_seconds_remaining",
    "yardline_100",
    "down",
    "ydstogo",
    "qtr",
    "posteam_timeouts_remaining",
    "defteam_timeouts_remaining",
    "posteam_spread",
]
for d in (train, val, test):
    d["posteam_type_home"] = (d["posteam_type"] == "home").astype(int)
PREDICTORS_FULL = PREDICTORS + ["posteam_type_home"]

X_train = train[PREDICTORS_FULL]
X_val = val[PREDICTORS_FULL]
X_test = test[PREDICTORS_FULL]

# ---------------------------------------------------------------------------
# Task 2: Single classification tree
# max_depth grid = {3,4,5,6,7}
# min node size grid = {1%,2%,3%,4%,5%} of training cardinality
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 2: Single Classification Tree")
print("=" * 70)

depth_grid = [3, 4, 5, 6, 7]
min_node_pct_grid = [0.01, 0.02, 0.03, 0.04, 0.05]

tree_results = []
for depth in depth_grid:
    for pct in min_node_pct_grid:
        min_node_size = max(1, int(round(pct * n_train)))
        clf = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=min_node_size,
            random_state=42,
        )
        clf.fit(X_train, y_train)
        val_pred = clf.predict_proba(X_val)[:, 1]
        ll = log_loss(y_val, val_pred)
        tree_results.append({
            "max_depth": depth,
            "min_node_pct": pct,
            "min_node_size": min_node_size,
            "val_log_loss": ll,
        })

tree_results_df = pd.DataFrame(tree_results).sort_values("val_log_loss").reset_index(drop=True)
print(tree_results_df.to_string(index=False))

best_tree_params = tree_results_df.iloc[0]
print(f"\nSelected: max_depth={int(best_tree_params['max_depth'])}, "
      f"min_node_pct={best_tree_params['min_node_pct']}, "
      f"min_node_size={int(best_tree_params['min_node_size'])}")

best_tree = DecisionTreeClassifier(
    max_depth=int(best_tree_params["max_depth"]),
    min_samples_leaf=int(best_tree_params["min_node_size"]),
    random_state=42,
)
best_tree.fit(X_train, y_train)
test_pred_tree = best_tree.predict_proba(X_test)[:, 1]
test_logloss_tree = log_loss(y_test, test_pred_tree)
print(f"Test log loss: {test_logloss_tree:.5f}")

# ---------------------------------------------------------------------------
# Task 3: Random forest
# n_estimators = 100 during tuning
# mtry (max_features) grid = {5,...,10}
# min node size grid = {1%,...,5%} of training cardinality
# max_depth grid = {3,...,7}
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 3: Random Forest")
print("=" * 70)

mtry_grid = [5, 6, 7, 8, 9, 10]
n_trees_tune = 100

rf_results = []
for mtry in mtry_grid:
    for depth in depth_grid:
        for pct in min_node_pct_grid:
            min_node_size = max(1, int(round(pct * n_train)))
            rf = RandomForestClassifier(
                n_estimators=n_trees_tune,
                max_features=mtry,
                max_depth=depth,
                min_samples_leaf=min_node_size,
                random_state=42,
                n_jobs=-1,
            )
            rf.fit(X_train, y_train)
            val_pred = rf.predict_proba(X_val)[:, 1]
            ll = log_loss(y_val, val_pred)
            rf_results.append({
                "mtry": mtry,
                "max_depth": depth,
                "min_node_pct": pct,
                "min_node_size": min_node_size,
                "val_log_loss": ll,
            })

rf_results_df = pd.DataFrame(rf_results).sort_values("val_log_loss").reset_index(drop=True)
print(rf_results_df.to_string(index=False))

best_rf_params = rf_results_df.iloc[0]
print(f"\nSelected: mtry={int(best_rf_params['mtry'])}, "
      f"max_depth={int(best_rf_params['max_depth'])}, "
      f"min_node_pct={best_rf_params['min_node_pct']}, "
      f"min_node_size={int(best_rf_params['min_node_size'])}")

# Final fit with increased number of trees
N_TREES_FINAL = 500
best_rf = RandomForestClassifier(
    n_estimators=N_TREES_FINAL,
    max_features=int(best_rf_params["mtry"]),
    max_depth=int(best_rf_params["max_depth"]),
    min_samples_leaf=int(best_rf_params["min_node_size"]),
    random_state=42,
    n_jobs=-1,
)
best_rf.fit(X_train, y_train)
test_pred_rf = best_rf.predict_proba(X_test)[:, 1]
test_logloss_rf = log_loss(y_test, test_pred_rf)
print(f"Test log loss (final fit, {N_TREES_FINAL} trees): {test_logloss_rf:.5f}")

# ---------------------------------------------------------------------------
# Task 4: XGBoost
# learning_rate grid = {0.1,...,0.5}
# max_depth grid = {3,...,7}
# min_child_weight grid = {1%,...,5%} of training cardinality
# num_boost_round grid = {2,3,4,5}, chosen via validation log loss (early stopping)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("TASK 4: XGBoost")
print("=" * 70)

lr_grid = [0.1, 0.2, 0.3, 0.4, 0.5]
min_child_weight_grid = [max(1, int(round(pct * n_train))) for pct in min_node_pct_grid]
max_rounds_grid = [2, 3, 4, 5]
max_num_boost_round = max(max_rounds_grid)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

xgb_results = []
for lr in lr_grid:
    for depth in depth_grid:
        for mcw_pct, mcw in zip(min_node_pct_grid, min_child_weight_grid):
            params = {
                "objective": "binary:logistic",
                "eval_metric": "logloss",
                "eta": lr,
                "max_depth": depth,
                "min_child_weight": mcw,
            }
            evals_result = {}
            booster = xgb.train(
                params,
                dtrain,
                num_boost_round=max_num_boost_round,
                evals=[(dval, "val")],
                evals_result=evals_result,
                verbose_eval=False,
            )
            val_losses_by_round = evals_result["val"]["logloss"]
            for n_rounds in max_rounds_grid:
                ll = val_losses_by_round[n_rounds - 1]
                xgb_results.append({
                    "learning_rate": lr,
                    "max_depth": depth,
                    "min_child_weight_pct": mcw_pct,
                    "min_child_weight": mcw,
                    "num_boost_round": n_rounds,
                    "val_log_loss": ll,
                })

xgb_results_df = pd.DataFrame(xgb_results).sort_values("val_log_loss").reset_index(drop=True)
print(xgb_results_df.to_string(index=False))

best_xgb_params = xgb_results_df.iloc[0]
print(f"\nSelected: learning_rate={best_xgb_params['learning_rate']}, "
      f"max_depth={int(best_xgb_params['max_depth'])}, "
      f"min_child_weight_pct={best_xgb_params['min_child_weight_pct']}, "
      f"min_child_weight={int(best_xgb_params['min_child_weight'])}, "
      f"num_boost_round={int(best_xgb_params['num_boost_round'])}")

final_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": best_xgb_params["learning_rate"],
    "max_depth": int(best_xgb_params["max_depth"]),
    "min_child_weight": int(best_xgb_params["min_child_weight"]),
}
final_booster = xgb.train(
    final_params,
    dtrain,
    num_boost_round=int(best_xgb_params["num_boost_round"]),
)
test_pred_xgb = final_booster.predict(dtest)
test_logloss_xgb = log_loss(y_test, test_pred_xgb)
print(f"Test log loss: {test_logloss_xgb:.5f}")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("SUMMARY: Test log loss by model")
print("=" * 70)
summary = pd.DataFrame({
    "model": ["Logistic", "Single Tree", "Random Forest", "XGBoost"],
    "test_log_loss": [test_logloss_logit, test_logloss_tree, test_logloss_rf, test_logloss_xgb],
})
print(summary.to_string(index=False))

Train rows: 117104, Val rows: 39034, Test rows: 38038

TASK 1: Logistic Regression (no tuning)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Validation log loss: 0.43004
Test log loss: 0.48041

TASK 2: Single Classification Tree
 max_depth  min_node_pct  min_node_size  val_log_loss
         7          0.05           5855      0.456094
         6          0.05           5855      0.456094
         5          0.05           5855      0.456094
         6          0.04           4684      0.456716
         7          0.04           4684      0.456716
         5          0.04           4684      0.456768
         4          0.05           5855      0.456910
         7          0.02           2342      0.457174
         6          0.02           2342      0.457211
         6          0.01           1171      0.457273
         5          0.01           1171      0.457447
         7          0.01           1171      0.457893
         5          0.02           2342      0.458480
         3          0.01           1171      0.459937
         3          0.05           5855      0.459937
         3          0.04           4684      0.4

In [2]:
# ---------------------------------------------------------------------------
# Final summary table: Model | Selected tuning parameters | Validation log loss | Test log loss
# ---------------------------------------------------------------------------
def fmt_params(d):
    return ", ".join(f"{k}={v}" for k, v in d.items())

logit_params = {}
tree_params = {
    "max_depth": int(best_tree_params["max_depth"]),
    "min_node_pct": f"{best_tree_params['min_node_pct']*100:.0f}%",
}
rf_params = {
    "mtry": int(best_rf_params["mtry"]),
    "max_depth": int(best_rf_params["max_depth"]),
    "min_node_pct": f"{best_rf_params['min_node_pct']*100:.0f}%",
    "n_trees": N_TREES_FINAL,
}
xgb_params = {
    "learning_rate": best_xgb_params["learning_rate"],
    "max_depth": int(best_xgb_params["max_depth"]),
    "min_child_weight_pct": f"{best_xgb_params['min_child_weight_pct']*100:.0f}%",
    "num_boost_round": int(best_xgb_params["num_boost_round"]),
}

val_logloss_tree = tree_results_df.iloc[0]["val_log_loss"]
val_logloss_rf = rf_results_df.iloc[0]["val_log_loss"]
val_logloss_xgb = xgb_results_df.iloc[0]["val_log_loss"]

final_table = pd.DataFrame([
    {
        "Model": "Logistic model",
        "Selected tuning parameters": fmt_params(logit_params) if logit_params else "N/A",
        "Validation log loss": round(val_logloss_logit, 5),
        "Test log loss": round(test_logloss_logit, 5),
    },
    {
        "Model": "Classification tree",
        "Selected tuning parameters": fmt_params(tree_params),
        "Validation log loss": round(val_logloss_tree, 5),
        "Test log loss": round(test_logloss_tree, 5),
    },
    {
        "Model": "Random forest",
        "Selected tuning parameters": fmt_params(rf_params),
        "Validation log loss": round(val_logloss_rf, 5),
        "Test log loss": round(test_logloss_rf, 5),
    },
    {
        "Model": "XGBoost",
        "Selected tuning parameters": fmt_params(xgb_params),
        "Validation log loss": round(val_logloss_xgb, 5),
        "Test log loss": round(test_logloss_xgb, 5),
    },
])

print("\n" + "=" * 100)
print("FINAL RESULTS TABLE")
print("=" * 100)
print(final_table.to_string(index=False))


FINAL RESULTS TABLE
              Model                                                 Selected tuning parameters  Validation log loss  Test log loss
     Logistic model                                                                        N/A              0.43004        0.48041
Classification tree                                               max_depth=7, min_node_pct=5%              0.45609        0.50587
      Random forest                          mtry=6, max_depth=7, min_node_pct=1%, n_trees=500              0.44427        0.48738
            XGBoost learning_rate=0.5, max_depth=7, min_child_weight_pct=1%, num_boost_round=5              0.44931        0.49265


In [ ]:
"""
Bootstrap-resampled logistic regression for win probability, with pointwise
95% confidence intervals, plotted three ways.

Training data: 2021-2024 (everything except 2025 test season).
100 bootstrap resamples of the training data, refit logistic each time.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

N_BOOT = 100
RNG = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# Load data, restrict to training window (2021-2024)
# ---------------------------------------------------------------------------
train = df[df["season"].between(2021, 2024)].copy()
train["posteam_type_home"] = (train["posteam_type"] == "home").astype(int)
n_train = len(train)
print(f"Training rows (2021-2024): {n_train}")

def build_logit_design(data):
    out = pd.DataFrame(index=data.index)
    out["score_differential"] = data["score_differential"]
    out["game_seconds_remaining"] = data["game_seconds_remaining"]
    out["score_diff_x_secs"] = data["score_differential"] * data["game_seconds_remaining"]
    out["yardline_100"] = data["yardline_100"]
    out["down"] = data["down"]
    out["yardline_x_down"] = data["yardline_100"] * data["down"]
    out["ydstogo"] = data["ydstogo"]
    out["qtr"] = data["qtr"]
    out["posteam_timeouts_remaining"] = data["posteam_timeouts_remaining"]
    out["defteam_timeouts_remaining"] = data["defteam_timeouts_remaining"]
    out["posteam_spread"] = data["posteam_spread"]
    out["posteam_type_home"] = data["posteam_type_home"]
    return out

y_train = train["posteam_win"].astype(int)
X_train_full = build_logit_design(train)

# ---------------------------------------------------------------------------
# Fit 100 bootstrap logistic models
# ---------------------------------------------------------------------------
boot_models = []
for b in range(N_BOOT):
    idx = RNG.integers(0, n_train, size=n_train)
    X_b = X_train_full.iloc[idx]
    y_b = y_train.iloc[idx]
    model = LogisticRegression(penalty=None, max_iter=2000)
    model.fit(X_b, y_b)
    boot_models.append(model)
    if (b + 1) % 20 == 0:
        print(f"Fitted bootstrap model {b + 1}/{N_BOOT}")

def predict_with_ci(design_df):
    """Return (median, lower 2.5%, upper 97.5%) predicted win prob across bootstrap models."""
    preds = np.column_stack([m.predict_proba(design_df)[:, 1] for m in boot_models])
    lower = np.percentile(preds, 2.5, axis=1)
    upper = np.percentile(preds, 97.5, axis=1)
    median = np.percentile(preds, 50, axis=1)
    return median, lower, upper

def make_grid_design(**kwargs):
    """Build a design matrix from a dict of column -> array/scalar, applying interactions."""
    n = max(len(v) if hasattr(v, "__len__") else 1 for v in kwargs.values())
    data = {}
    for k, v in kwargs.items():
        data[k] = np.full(n, v) if not hasattr(v, "__len__") else np.asarray(v)
    grid = pd.DataFrame(data)
    out = pd.DataFrame(index=grid.index)
    out["score_differential"] = grid["score_differential"]
    out["game_seconds_remaining"] = grid["game_seconds_remaining"]
    out["score_diff_x_secs"] = grid["score_differential"] * grid["game_seconds_remaining"]
    out["yardline_100"] = grid["yardline_100"]
    out["down"] = grid["down"]
    out["yardline_x_down"] = grid["yardline_100"] * grid["down"]
    out["ydstogo"] = grid["ydstogo"]
    out["qtr"] = grid["qtr"]
    out["posteam_timeouts_remaining"] = grid["posteam_timeouts_remaining"]
    out["defteam_timeouts_remaining"] = grid["defteam_timeouts_remaining"]
    out["posteam_spread"] = grid["posteam_spread"]
    out["posteam_type_home"] = grid["posteam_type_home"]
    return out, grid

# Common fixed values shared across plots 1 and 2
COMMON = dict(
    yardline_100=50,
    ydstogo=10,
    posteam_timeouts_remaining=3,
    defteam_timeouts_remaining=3,
    posteam_spread=0,
    posteam_type_home=1,
)

# ===========================================================================
# PLOT 1: win prob vs score_differential, colored by quarter, faceted by down
# fixed: game_seconds_remaining = 2400, plus COMMON
# ===========================================================================
score_diff_range = np.arange(-28, 29, 1)
downs = [1, 2, 3, 4]
quarters = [1, 2, 3, 4]

rows = []
for down in downs:
    for qtr in quarters:
        design, grid = make_grid_design(
            score_differential=score_diff_range,
            game_seconds_remaining=2400,
            down=down,
            qtr=qtr,
            **COMMON,
        )
        median, lower, upper = predict_with_ci(design)
        rows.append(pd.DataFrame({
            "score_differential": score_diff_range,
            "down": down,
            "qtr": qtr,
            "median": median,
            "lower": lower,
            "upper": upper,
        }))
plot1_df = pd.concat(rows, ignore_index=True)

fig1, axes1 = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(quarters)))
for ax, down in zip(axes1, downs):
    sub_down = plot1_df[plot1_df["down"] == down]
    for qtr, color in zip(quarters, colors):
        sub = sub_down[sub_down["qtr"] == qtr]
        ax.plot(sub["score_differential"], sub["median"], color=color, label=f"Q{qtr}")
        ax.fill_between(sub["score_differential"], sub["lower"], sub["upper"], color=color, alpha=0.15)
    ax.set_title(f"Down = {down}")
    ax.set_xlabel("Score differential")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes1[0].set_ylabel("Predicted win probability")
axes1[0].legend(title="Quarter")
fig1.suptitle("Win Probability vs Score Differential (95% bootstrap CI), by Down/Quarter")
fig1.tight_layout()

# ===========================================================================
# PLOT 2: heatmap of win prob, score_differential (x) vs game_seconds_remaining (y)
# faceted by down. fixed: COMMON
# ===========================================================================
score_diff_grid = np.arange(-21, 22, 3)
secs_grid = np.arange(0, 3601, 120)

fig2, axes2 = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
for ax, down in zip(axes2, downs):
    Z = np.zeros((len(secs_grid), len(score_diff_grid)))
    for i, secs in enumerate(secs_grid):
        design, grid = make_grid_design(
            score_differential=score_diff_grid,
            game_seconds_remaining=secs,
            down=down,
            qtr=1,  # qtr not used as an axis here; held arbitrary, doesn't affect heatmap geometry
            **COMMON,
        )
        median, _, _ = predict_with_ci(design)
        Z[i, :] = median
    im = ax.imshow(
        Z,
        aspect="auto",
        origin="lower",
        extent=[score_diff_grid.min(), score_diff_grid.max(), secs_grid.min(), secs_grid.max()],
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
    )
    ax.set_title(f"Down = {down}")
    ax.set_xlabel("Score differential")
axes2[0].set_ylabel("Game seconds remaining")
fig2.colorbar(im, ax=axes2, label="Predicted win probability", fraction=0.02, pad=0.02)
fig2.suptitle("Win Probability Heatmap: Score Differential vs Game Seconds Remaining, by Down")

# ===========================================================================
# PLOT 3: win prob vs yardline_100, for score_differential in {-7,0,7}, facet by down
# fixed: qtr=1, ydstogo=10, timeouts=3/3, spread=0, home, game_seconds_remaining=2400
# ===========================================================================
yardline_range = np.arange(1, 100, 1)
score_diffs_p3 = [-7, 0, 7]

rows3 = []
for down in downs:
    for sd in score_diffs_p3:
        design, grid = make_grid_design(
            yardline_100=yardline_range,
            score_differential=sd,
            game_seconds_remaining=2400,
            down=down,
            qtr=1,
            ydstogo=10,
            posteam_timeouts_remaining=3,
            defteam_timeouts_remaining=3,
            posteam_spread=0,
            posteam_type_home=1,
        )
        median, lower, upper = predict_with_ci(design)
        rows3.append(pd.DataFrame({
            "yardline_100": yardline_range,
            "down": down,
            "score_differential": sd,
            "median": median,
            "lower": lower,
            "upper": upper,
        }))
plot3_df = pd.concat(rows3, ignore_index=True)

fig3, axes3 = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
sd_colors = {-7: "tab:red", 0: "tab:gray", 7: "tab:green"}
for ax, down in zip(axes3, downs):
    sub_down = plot3_df[plot3_df["down"] == down]
    for sd in score_diffs_p3:
        sub = sub_down[sub_down["score_differential"] == sd]
        ax.plot(sub["yardline_100"], sub["median"], color=sd_colors[sd], label=f"Score diff = {sd}")
        ax.fill_between(sub["yardline_100"], sub["lower"], sub["upper"], color=sd_colors[sd], alpha=0.15)
    ax.set_title(f"Down = {down}")
    ax.set_xlabel("Yardline (100 = own goal line)")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes3[0].set_ylabel("Predicted win probability")
axes3[0].legend()
fig3.suptitle("Win Probability vs Yardline_100 (95% bootstrap CI), by Down/Score Differential")
fig3.tight_layout()

plt.show()

Training rows (2021-2024): 156138


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

Fitted bootstrap model 20/100


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

Fitted bootstrap model 40/100


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

Fitted bootstrap model 60/100


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a

Fitted bootstrap model 80/100


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please a